##***Problem Definition: Applying dynamic programming algorithms, such as policy evaluation and policy***

In [ ]:
#Iqra Kaularikar | 221A060 | 14
!pip install gym

In [ ]:
#Iqra Kaularikar | 221A060 | 14
!pip install gridworld

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 1.9 MB/s eta 0:00:00
  Created wheel for gridworld: filename=gridworld-0.1-py3-none-any.whl size=4319 sha256=dc3eb6f49f651c53cce3d61818f435513f7a9c66747a4317f525e2c20ca85ac0
  Stored in directory: /root/.cache/pip/wheels/cf/39/dd/3ee6d82ac47034568c281698d7054f38c092bcacf1a71b908f
Successfully built gridworld


In [ ]:
#Iqra Kaularikar | 221A060 | 14
import gymnasium as gym
import numpy as np
env = gym.make("FrozenLake-v1")

In [ ]:
#Iqra Kaularikar | 221A060 | 14
def policy_evaluation(policy, environment, discount_factor=1.0, theta=1e-9, max_iterations=1e9):

    #Initialize the value function for each state to zero
    V = np.zeros(environment.observation_space.n)
    # Number of evaluation iterations
    evaluation_iterations = 1

    for i in range(int(max_iterations)):
        # Initialize a change of value function as zero for the current iteration
        delta = 0

        # Iterate through each state
        for state in range(environment.observation_space.n):
            # Initialize a new value of the current state
            v = 0

            # Try all possible actions which can be taken from this state
            for action, action_probability in enumerate(policy[state]):
                # Check how good the next state will be
                for state_probability, next_state, reward, terminated in environment.P[state][action]:
                    # Calculate the expected value
                    # Corrected `.` to `*` for multiplication and discount factor application
                    v += action_probability * state_probability * (reward + discount_factor * V[next_state])

            # Calculate the absolute change of value function
            delta = max(delta, np.abs(V[state] - v))
            # Update value function
            V[state] = v

        evaluation_iterations += 1

        #Terminate if value change is significant after an entire sweep
        if delta < theta:
            print(f'Policy evaluated in {evaluation_iterations} iterations.')
            return V

    # This part executes if max_iterations is reached without convergence
    print(f'Maximum iterations reached. Policy evaluation terminated after {evaluation_iterations} iterations.')
    return V

In [ ]:
#Iqra Kaularikar | 221A060 | 14
def one_step_lookahead(environment, state, V, discount_factor):
    action_values = np.zeros(environment.action_space.n)
    for action in range(environment.action_space.n):
        for probability, next_state, reward, terminated in environment.P[state][action]:
            action_values[action] += probability * (reward + discount_factor * V[next_state])
    return action_values

In [ ]:
#Iqra Kaularikar | 221A060 | 14
def policy_iteration(environment, discount_factor=1.0, max_iterations=1e9):
         # Start with a random policy
         #num states x num actions / num actions
         policy = np.ones([environment.observation_space.n, environment.action_space.n]) / environment.action_space.n
         # Initialize counter of evaluated policies
         evaluated_policies = 1
         # Repeat until convergence or critical number of iterations reached
         for i in range(int(max_iterations)):
                 stable_policy = True
                 # Evaluate current policy
                 V = policy_evaluation(policy, environment, discount_factor=discount_factor)
                 # Go through each state and try to improve (policy improvement)
                 for state in range(environment.observation_space.n):
                         # Choose the best action in a current state under current policy
                         current_action = np.argmax(policy[state])
                         # Look one step ahead and evaluate if current action is optimal
                         # We will try every possible action in a current state
                         action_value = one_step_lookahead(environment, state, V, discount_factor)
                         # Select a better action
                         best_action = np.argmax(action_value)
                         # If action didn't change
                         if current_action != best_action:
                                stable_policy = True
                                # Greedy policy update
                                policy[state] = np.eye(environment.action_space.n)[best_action]
                 evaluated_policies += 1
                 # If the algorithm converged and policy is not changing anymore, then return final policy and value function
                 if stable_policy:
                        print(f'Evaluated {evaluated_policies} policies.')
                        return policy, V

In [ ]:
#Iqra Kaularikar | 221A060 | 14
def value_iteration(environment, discount_factor=1.0, theta=1e-9, max_iterations=1e9):
    # Initialize state-value function with zeros for each environment state
    V = np.zeros(environment.observation_space.n)
    for i in range(int(max_iterations)):
        # Early stopping condition
        delta = 0
        # Update each state
        for state in range(environment.observation_space.n): # Corrected from action_space.n
            # Do a one-step lookahead to calculate state-action values
            action_value = one_step_lookahead(environment, state, V, discount_factor)
            # Select best action to perform based on the highest state-action value
            best_action_value = np.max(action_value)
            # Calculate change in value
            delta = max(delta, np.abs(V[state] - best_action_value))
            # Update the value function for current state
            V[state] = best_action_value
        # Check if we can stop
        if delta < theta:
            print(f'Value-iteration converged at iteration#{i}.')
            break

    # Create a deterministic policy using the optimal value function
    policy = np.zeros([environment.observation_space.n, environment.action_space.n])
    for state in range(environment.observation_space.n):
        # One step lookahead to find the best action for this state
        action_value = one_step_lookahead(environment, state, V, discount_factor)
        # Select best action based on the highest state-action value
        best_action = np.argmax(action_value)
        # Update the policy to perform a better action at a current state
        policy[state, best_action] = 1.0
    return policy, V

In [ ]:
#Iqra Kaularikar | 221A060 | 14
def play_episodes(environment, n_episodes, policy):
    wins = 0
    total_reward = 0
    for episode in range(n_episodes):
        terminated = False
        truncated = False # Add truncated flag for Gymnasium compatibility
        state, info = environment.reset() # Unpack state from (observation, info) tuple
        while not terminated and not truncated:
            # Select best action to perform in a current state
            action = np.argmax(policy[state])
            # Perform an action an observe how environment acted in response
            next_state, reward, terminated, truncated, info = environment.step(action)
            # Summarize total reward
            total_reward += reward
            # Update current state
            state = next_state
            # Calculate number of wins over episodes
            if terminated and reward == 1.0:
                wins += 1
    average_reward = total_reward / n_episodes
    return wins, total_reward, average_reward

In [ ]:
#Iqra Kaularikar | 221A060 | 14
# Number of episodes to play
n_episodes = 10000
# Functions to find best policy
solvers = [('Policy Iteration', policy_iteration),
           ('Value Iteration', value_iteration)]
for iteration_name, iteration_func in solvers:
    # Load a Frozen Lake environment
    environment = gym.make('FrozenLake-v1', render_mode = "rgb_array")
    # Search for an optimal policy using policy iteration
    policy, V = iteration_func(environment.unwrapped)
    # Apply best policy to the real environment
    wins, total_reward, average_reward = play_episodes(environment, n_episodes, policy)
    print(f'{iteration_name} :: number of wins over {n_episodes} episodes = {wins}')
    print(f'{iteration_name} :: average reward over {n_episodes} episodes = {average_reward} \n\n')

Policy evaluated in 66 iterations.
Evaluated 2 policies.
Policy Iteration :: number of wins over 10000 episodes = 7337
Policy Iteration :: average reward over 10000 episodes = 0.7337 


Value-iteration converged at iteration#523.
Value Iteration :: number of wins over 10000 episodes = 7361
Value Iteration :: average reward over 10000 episodes = 0.7361 


